In [ ]:
import copy, random
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
 
device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)

In [ ]:
tfm = transforms.ToTensor()
full_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=tfm)
test_ds    = datasets.FashionMNIST("./data", train=False, download=True, transform=tfm)
n_unused = len(full_train) - 6000 - 1000
train_ds, val_ds, _ = random_split(
    full_train, [6000, 1000, n_unused],
    generator=torch.Generator().manual_seed(SEED)
)

def make_loader(ds, shuffle, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=128, shuffle=shuffle, generator=g if shuffle else None)

val_loader  = make_loader(val_ds,  False)
test_loader = make_loader(test_ds, False)
print(len(train_ds), len(val_ds), len(test_ds))

print(torch.Generator().manual_seed(SEED))

In [ ]:
def build_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 10),
    )

# One set of initial weights, shared by both runs
set_seed(SEED)
init_state = copy.deepcopy(build_model().state_dict())

hard_loss = nn.CrossEntropyLoss(label_smoothing=0.0, reduction="sum")
soft_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def evaluate(model, loader):
    """Returns hard-loss CE (mean per sample), accuracy, and all confidences."""
    model.eval()
    tot_loss, correct, n, confs = 0.0, 0, 0, []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        tot_loss += hard_loss(logits, y).item()
        correct  += (logits.argmax(-1) == y).sum().item()
        n += y.size(0)
        confs.append(torch.softmax(logits, dim=-1).max(dim=-1).values.cpu())
    return tot_loss / n, correct / n, torch.cat(confs).numpy()

def train_model(train_criterion, epochs=10):
    model = build_model().to(device)
    model.load_state_dict(init_state)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    train_loader = make_loader(train_ds, True, SEED)   # same shuffle order for both runs

    best_val, best_epoch, best_state = float("inf"), None, None
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            train_criterion(model(x), y).backward()
            opt.step()

        val_loss, val_acc, _ = evaluate(model, val_loader)   # validation always uses hard CE
        print(f"epoch {ep:2d} | val CE {val_loss:.4f} | val acc {val_acc:.4f}")
        if val_loss < best_val:          # strict '<' keeps the earlier epoch on ties
            best_val, best_epoch = val_loss, ep
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)    # restore best checkpoint before testing
    return model, best_epoch

In [ ]:
results, test_confs = {}, {}
for name, crit in [("Hard (ls=0.0)", hard_loss), ("Soft (ls=0.1)", soft_loss)]:
    print(f"\n=== {name} ===")
    model, best_ep = train_model(crit, epochs=10)
    _, test_acc, confs = evaluate(model, test_loader)
    results[name] = dict(epoch=best_ep, acc=test_acc, conf=confs.mean())
    test_confs[name] = confs

print("\n{:<16}{:>16}{:>15}{:>20}".format("Model", "Selected epoch", "Test acc", "Mean test conf"))
for k, v in results.items():
    print("{:<16}{:>16d}{:>15.4f}{:>20.4f}".format(k, v["epoch"], v["acc"], v["conf"]))

In [ ]:
bins = np.linspace(0, 1, 21)   # 20 equal-width bins from 0 to 1
plt.figure(figsize=(7, 4.5))
plt.hist(test_confs["Hard (ls=0.0)"], bins=bins, alpha=0.55, label="Hard loss (ls=0.0)")
plt.hist(test_confs["Soft (ls=0.1)"], bins=bins, alpha=0.55, label="Soft loss (ls=0.1)")
plt.xlabel("Confidence (max softmax probability)")
plt.ylabel("Number of test images")
plt.title("FashionMNIST test confidence")
plt.legend()
plt.tight_layout()
plt.show()

Label smoothing left accuracy essentially unchanged (82.31% hard vs 82.48% soft, both selected at epoch 8, a 0.17-point gap that is within noise for one seed) but clearly lowered confidence: mean test confidence fell from 0.807 to 0.7145. The histogram shows why: the hard-loss model piles about 3,800 test images into the top bin (0.95 to 1.0), while the soft-loss model has only about 1,200 there and shifts mass into the 0.8 to 0.95 range. With hard targets, cross-entropy keeps falling as the true-class probability approaches 1, so the gradient always pushes the logit gap larger. With a 0.91/0.01 target, the loss is minimized when the model outputs about 0.91 for the true class, and pushing beyond that increases the loss because the 0.01 on the other classes is penalized. That puts a ceiling on useful confidence, so the model has no incentive to produce the extreme logits that dominate the hard-loss histogram